# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [50]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [51]:
import pandas as pd

In [52]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
885,This magazine has too many advertisements and ...,bad
527,far too many adds not enought info,bad
126,This is the one magazine that makes affordable...,good
50,Husband enjoying his favorite magazine.,good
783,"In my opinion, it doesn't have alot of content...",bad


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [53]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [54]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [55]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [56]:
from sklearn import metrics

In [57]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [58]:
evaluate(sgd_clf)

Accuracy: 76.1%


## Assignment Task 1


In [59]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compare_models(models, X_train, y_train, X_test, y_test):
    results = []

    if isinstance(models, list):
        models = {type(model).__name__: model for model in models}

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results.append({
            'Model name': model_name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, pos_label='good'),
            'Recall': recall_score(y_test, y_pred, pos_label='good'),
            'F1-score': f1_score(y_test, y_pred, pos_label='good')
        })

    return pd.DataFrame(results)

In [60]:
# Testing with the SGD baseline
sgd_pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

results = compare_models({'SGD Baseline': sgd_pipeline},
                        train['review'], train['label'],
                        test['review'], test['label'])
results

,Model name,Accuracy,Precision,Recall,F1-score
0,SGD Baseline,0.765,0.751423,0.792,0.771178


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [61]:
# YOUR CODE HERE
# using Naive Bayed classifier as the second model
from sklearn.naive_bayes import MultinomialNB

mnb_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

_ = mnb_clf.fit(train['review'], train['label'])


# evaluate your model and see if it does better
# than the ones we provided

def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

evaluate(mnb_clf)

Accuracy: 85.3%


In [62]:
# Compare both models together
models_to_compare = {
    'SGD Baseline': sgd_pipeline,  # from your test cell
    'Naive Bayes': mnb_clf         # your NB model
}

results_comparison = compare_models(models_to_compare,
                                   train['review'], train['label'],
                                   test['review'], test['label'])
results_comparison

,Model name,Accuracy,Precision,Recall,F1-score
0,SGD Baseline,0.764,0.764000,0.764,0.764000
1,Naive Bayes,0.853,0.853707,0.852,0.852853


#### My model (MultinomialNP) performed better than SGDClassifier based on the evaluation metrics

## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [63]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [64]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [65]:
# YOUR CODE HERE
for i in range(10):
    print(train.iloc[i].to_dict())


{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}
{'review': "I still have not received this.  Obviously I can't review something I haven't seen.  Where is my order???????", 'label': 'bad'}
{'review': '</tr>The magazine is not worth the cost of subscription.</tr>', 'label': 'good'}
{'review': 'This magazine is basically ads. Kindve worthless to me really.', 'label': 'bad'}
{'review': "The only thing I've recieved, so far, is the bill.\nI know this is [ or was when I last subscribed ] a ten month magazine - but so far not even the usual promotional papers.......", 'label': 'bad'}
{'review': 'The magazines are great, but I never received the golf shoe bag that was supposed to accompany my subscription...<body>', 'label': 'good'}
{'review': 'This is one magazine I really love. It has primitive country decor and the a

## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [66]:
import re

def is_bad_data(review: str) -> bool:
    if pd.isna(review):
        return True
    return '<br' in review.lower() or bool(re.search(r'<[^>]+>', review))

## Creating the cleaned training set

In [67]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [68]:
from sklearn import clone

In [69]:
sgd_clf_clean = clone(sgd_clf)

In [70]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [71]:
evaluate(sgd_clf_clean)

Accuracy: 97.2%


In [73]:
# After sgd_clf_clean evaluation cell:
print("✅ Data cleaning improved SGD from 76.1% → 97.2% accuracy!")
print(f"Cleaned dataset size: {len(train_clean)} / {len(train)} ({100*len(train_clean)/len(train):.1f}% kept)")

✅ Data cleaning improved SGD from 76.1% → 97.2% accuracy!
Cleaned dataset size: 4018 / 6666 (60.3% kept)
